In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="prc-data-challenge-2025")


def q(sql):
    return client.query(sql).to_dataframe()

In [ ]:
# preview 10 rows
q("""
SELECT *
FROM `prc-data-challenge-2025.prc2025.fuel_rank`
LIMIT 10
""")

Same columns as fuel_train and fuel_kg is filled, so the true values of the rank set are public.
First interval is on 2025-09-01, after the training period.

In [ ]:
# preview 10 rows
q("""
SELECT *
FROM `prc-data-challenge-2025.prc2025.fuel_final`
LIMIT 10
""")

fuel_final has fuel_kg too, starting 2025-10-01.
This flight looks different: short intervals (40 s to 4 min), gaps between them, and values in pounds (8.164656 kg = 18 lb).

In [ ]:
# preview 10 rows
q("""
SELECT *
FROM `prc-data-challenge-2025.prc2025.fuel_submit_rank`
LIMIT 10
""")

Same intervals as fuel_rank but fuel_kg is empty. It's the file to fill for a submission.

In [ ]:
# preview 10 rows
q("""
SELECT *
FROM `prc-data-challenge-2025.prc2025.fuel_submit_final`
LIMIT 10
""")

fuel_submit_final starts with the rank intervals, so idx 0 here is not idx 0 in fuel_final. To score on final we'll have to join on flight_id, start and end.

In [ ]:
q(
    """
SELECT 'fuel_rank' AS table_name, COUNT(*) AS n_rows, COUNT(DISTINCT flight_id) AS n_flights
FROM `prc-data-challenge-2025.prc2025.fuel_rank`
UNION ALL
SELECT 'fuel_submit_rank', COUNT(*), COUNT(DISTINCT flight_id)
FROM `prc-data-challenge-2025.prc2025.fuel_submit_rank`
UNION ALL
SELECT 'fuel_final', COUNT(*), COUNT(DISTINCT flight_id)
FROM `prc-data-challenge-2025.prc2025.fuel_final`
UNION ALL
SELECT 'fuel_submit_final', COUNT(*), COUNT(DISTINCT flight_id)
FROM `prc-data-challenge-2025.prc2025.fuel_submit_final`
"""
)

Counts match the website. fuel_submit_final is rank + final (61,745 intervals, 4,724 flights), no flight appears in both.